In [ ]:
# ============================================================
# GOOGLE DRIVE MOUNT
# ============================================================
# Run this cell first in Google Colab.
# After it runs, your Google Drive should appear at:
# /content/drive/MyDrive

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Google Drive mount skipped or failed.')
    print('If you are not using Colab, this is normal.')
    print('Error:', e)

from pathlib import Path

print('Check /content/drive exists:', Path('/content/drive').exists())
print('Check /content/drive/MyDrive exists:', Path('/content/drive/MyDrive').exists())

if Path('/content/drive/MyDrive').exists():
    print('Top folders in MyDrive:')
    for p in list(Path('/content/drive/MyDrive').iterdir())[:30]:
        print(' -', p.name)
else:
    print('MyDrive not found yet. In Colab, click the Google Drive authorization link and allow access.')


# Train EfficientNet-B0 Model

FYP2 training notebook. This replaces FYP1 ensemble training with **one EfficientNet-B0 model**.

Training labels still using binary:

- `real` = 0
- `fake` = 1

The Flask system able to converts model probability into three output states later: `REAL`, `FAKE`, or `UNCERTAIN`.


In [ ]:
from pathlib import Path
import os, random, shutil, zipfile

# ============================================================
# DATASET CONFIGURATION
# ============================================================
# Reverted to training one dataset only...
DRIVE_ZIP_PATHS = [
    Path('/content/drive/MyDrive/dataset.zip')
]
EXTRACT_DIR = Path('/content/Dataset_extracted')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob('*') if p.suffix.lower() in IMAGE_EXTS)

def extract_all_datasets():
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    found_any = False
    for zip_path in DRIVE_ZIP_PATHS:
        if not zip_path.exists():
            print(f'Warning: {zip_path} not found. Skipping...')
            continue

        found_any = True
        print(f'Extracting {zip_path}...')
        temp_sub = EXTRACT_DIR / f"temp_{zip_path.stem}_{random.randint(0,999)}"
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(temp_sub)

        for item in temp_sub.rglob('*'):
            if item.is_file() and item.suffix.lower() in IMAGE_EXTS:
                parts = item.parts
                relevant_parts = []
                found = False
                for p in parts:
                    if p.lower() in ['train', 'val', 'validation', 'test', 'real', 'fake']:
                        found = True
                    if found:
                        relevant_parts.append(p)

                if relevant_parts:
                    dest = EXTRACT_DIR / Path(*relevant_parts)
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    if dest.exists():
                        dest = dest.parent / f"{zip_path.stem}_{item.name}"
                    shutil.move(str(item), str(dest))

        shutil.rmtree(temp_sub)

    if not found_any:
        print('Error: dataset.zip was not found in MyDrive.')
    else:
        print('Extraction completed.')
        print('Total image count:', count_images(EXTRACT_DIR))

def find_class_folder(parent, names):
    parent = Path(parent)
    for name in names:
        p = parent / name
        if p.exists() and p.is_dir():
            return p
    return None

def has_train_val_structure(root):
    root = Path(root)
    train_dir = find_class_folder(root, ['train', 'Train', 'TRAIN', 'training', 'Training'])
    val_dir = find_class_folder(root, ['val', 'Val', 'VAL', 'valid', 'Valid', 'validation', 'Validation'])
    if not train_dir or not val_dir: return False, None, None
    return True, train_dir, val_dir

def resolve_dataset_dir():
    if not Path('/content/drive/MyDrive').exists():
        raise FileNotFoundError('Mount Google Drive first.')

    extract_all_datasets()
    return EXTRACT_DIR

DATASET_DIR = resolve_dataset_dir()
print('\nDATASET_DIR =', DATASET_DIR)
ok, t_dir, v_dir = has_train_val_structure(DATASET_DIR)
if ok:
    print('Train images:', count_images(t_dir))
    print('Validation images:', count_images(v_dir))
else:
    print('Total images found:', count_images(DATASET_DIR))

Extracting /content/drive/MyDrive/dataset.zip...
Extraction completed.
Total image count: 59944

DATASET_DIR = /content/Dataset_extracted
Train images: 47956
Validation images: 5994


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def build_model(pretrained=True):
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    return model

# Assuming DEVICE, LR are defined elsewhere or need to be define
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LR = 1e-3

model = build_model(pretrained=True).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 140MB/s]


In [ ]:
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, labels in loader:
            x = x.to(DEVICE)
            y = remap(labels, loader.dataset.class_to_idx).numpy().astype(int).tolist()
            logits = model(x)
            probs = torch.sigmoid(logits).view(-1).cpu().numpy()
            preds = (probs >= 0.5).astype(int).tolist()
            y_true.extend(y)
            y_pred.extend(preds)
    return {
        'accuracy': round(accuracy_score(y_true, y_pred)*100, 2),
        'precision_fake': round(precision_score(y_true, y_pred, zero_division=0)*100, 2),
        'recall_fake': round(recall_score(y_true, y_pred, zero_division=0)*100, 2),
        'f1_fake': round(f1_score(y_true, y_pred, zero_division=0)*100, 2),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0,1]).tolist()
    }


In [ ]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from PIL import Image, UnidentifiedImageError

# === Configuration ===
EPOCHS = 10
BATCH_SIZE = 32
IMG_SIZE = 224

# Drive storage configuration
DRIVE_SAVE_PATH = Path('/content/drive/MyDrive/fyp2_checkpoints')
DRIVE_SAVE_PATH.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = DRIVE_SAVE_PATH / 'efficientnet_b0_best.pth'

# Remapping function
def remap(labels, class_to_idx_map=None):
    return 1 - labels.float()

# Image transformations
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Custom function to check if an image file is valid
def is_valid_image(path):
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            img.verify() # Verify that it is, indeed, an image
            return True
    except (IOError, SyntaxError, UnidentifiedImageError) as e:
        # print(f"Skipping corrupted image: {path} - {e}") # Uncomment to see skipped files
        return False

train_ds_path = DATASET_DIR / 'Train'
val_ds_path = DATASET_DIR / 'Validation'

train_ds = datasets.ImageFolder(train_ds_path, transform=train_tf, is_valid_file=is_valid_image)
val_ds = datasets.ImageFolder(val_ds_path, transform=val_tf, is_valid_file=is_valid_image)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

best_f1 = -1

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for x, labels in train_loader:
        x = x.to(DEVICE)
        y = remap(labels, train_ds.class_to_idx).to(DEVICE)
        optimizer.zero_grad()
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Evaluation
    metrics = evaluate(model, val_loader)
    current_f1 = metrics['f1_fake']
    avg_loss = total_loss / len(train_loader)

    print(f'Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {current_f1}%')

    # Save every epoch to Drive
    epoch_save_path = DRIVE_SAVE_PATH / f'model_epoch_{epoch}.pth'
    torch.save(model.state_dict(), epoch_save_path)

    # Save the BEST model exclusively
    if current_f1 > best_f1:
        best_f1 = current_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'---> New best model saved to Drive: {BEST_MODEL_PATH}')

print('\nTraining Complete.')
print(f'Best F1 Score achieved: {best_f1}%')

Epoch 1/10 | Loss: 0.2010 | Val F1: 93.63%
---> New best model saved to Drive: /content/drive/MyDrive/fyp2_checkpoints/efficientnet_b0_best.pth
Epoch 2/10 | Loss: 0.1749 | Val F1: 96.84%
---> New best model saved to Drive: /content/drive/MyDrive/fyp2_checkpoints/efficientnet_b0_best.pth
Epoch 3/10 | Loss: 0.1548 | Val F1: 95.35%
Epoch 4/10 | Loss: 0.1427 | Val F1: 96.53%
Epoch 5/10 | Loss: 0.1356 | Val F1: 96.53%
Epoch 6/10 | Loss: 0.1283 | Val F1: 96.8%
Epoch 7/10 | Loss: 0.1211 | Val F1: 97.66%
---> New best model saved to Drive: /content/drive/MyDrive/fyp2_checkpoints/efficientnet_b0_best.pth
Epoch 8/10 | Loss: 0.1097 | Val F1: 96.9%
Epoch 9/10 | Loss: 0.1106 | Val F1: 96.83%
Epoch 10/10 | Loss: 0.1046 | Val F1: 97.26%

Training Complete.
Best F1 Score achieved: 97.66%


In [ ]:
# Optional: test folder evaluation if dataset/test exists
test_dir = DATASET_DIR / 'test'
if test_dir.exists():
    test_ds = datasets.ImageFolder(test_dir, transform=val_tf)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print('Test metrics:', evaluate(model, test_loader))
else:
    print('No test folder found. Skipped test evaluation.')
